In [ ]:
import xlrd
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output, callback
import dash_bootstrap_components as dbc

In [ ]:
# Country name mapping (used across all datasets)
COUNTRY_MAP = {
    'AU:Australia'       : 'Australia',
    'CA:Canada'          : 'Canada',
    'CH:Switzerland'     : 'Switzerland',
    'GB:United Kingdom'  : 'United Kingdom',
    'JP:Japan'           : 'Japan',
    'US:United States'   : 'United States',
    'XM:Euro area'       : 'Euro Area',
    'United States'      : 'United States',
    'United Kingdom'     : 'United Kingdom',
    'European Union (27)': 'Euro Area',
    'Japan'              : 'Japan',
    'Australia'          : 'Australia',
    'Canada'             : 'Canada',
    'Switzerland'        : 'Switzerland',
    'Euro area'          : 'Euro Area',
}

COUNTRIES = ['Australia', 'Canada', 'Switzerland',
             'United Kingdom', 'Japan', 'United States', 'Euro Area']

#  INFLATION (CPI YoY % change) 

In [ ]:
cpi_raw = pd.read_excel('consumer_prices.xlsx', sheet_name='timeseries observations')

inflation = (
    cpi_raw[cpi_raw['Unit'] == 'Year-on-year changes, in per cent']
    [['REF_AREA:Reference area', 'TIME_PERIOD:Period', 'OBS_VALUE:Value']]
    .rename(columns={
        'REF_AREA:Reference area' : 'Country',
        'TIME_PERIOD:Period'      : 'Date',
        'OBS_VALUE:Value'         : 'Inflation'
    })
)
inflation['Country'] = inflation['Country'].map(COUNTRY_MAP)
inflation['Date'] = pd.to_datetime(inflation['Date'])
inflation = inflation[inflation['Country'].isin(COUNTRIES)].reset_index(drop=True)

#  POLICY RATES

In [ ]:
rates_raw = pd.read_excel('policy_rates.xlsx', sheet_name='timeseries observations')

rates = (
    rates_raw
    [['REF_AREA:Reference area', 'TIME_PERIOD:Period', 'OBS_VALUE:Value']]
    .rename(columns={
        'REF_AREA:Reference area' : 'Country',
        'TIME_PERIOD:Period'      : 'Date',
        'OBS_VALUE:Value'         : 'Policy_Rate'
    })
)
rates['Country'] = rates['Country'].map(COUNTRY_MAP)
rates['Date'] = pd.to_datetime(rates['Date'])
rates = rates[rates['Country'].isin(COUNTRIES)].reset_index(drop=True)

#   UNEMPLOYMENT (annual → monthly via forward fill)

In [ ]:
unemp_raw = pd.read_csv('unemployment-rate.csv')

unemp = (
    unemp_raw
    .rename(columns={
        'Entity'            : 'Country',
        'Year'              : 'Year',
        'Unemployment rate' : 'Unemployment'
    })
    [['Country', 'Year', 'Unemployment']]
)
unemp['Country'] = unemp['Country'].map(COUNTRY_MAP).fillna(unemp['Country'])
unemp = unemp[unemp['Country'].isin(COUNTRIES)]

# Expand annual to monthly dates (Jan of each year, then forward fill)
unemp['Date'] = pd.to_datetime(unemp['Year'].astype(int).astype(str) + '-01-01')
unemp = unemp[['Country', 'Date', 'Unemployment']]

# Create full monthly date range per country and forward fill
monthly_range = pd.date_range('2007-01-01', '2026-03-01', freq='MS')
unemp_monthly = []
for country in COUNTRIES:
    df_c = unemp[unemp['Country'] == country].set_index('Date').reindex(monthly_range)
    df_c['Country'] = country
    df_c['Unemployment'] = df_c['Unemployment'].ffill()
    df_c.index.name = 'Date'
    unemp_monthly.append(df_c.reset_index())

unemp_monthly = pd.concat(unemp_monthly, ignore_index=True)

#   GDP GROWTH (annual → monthly via forward fill + forecast flag)

In [ ]:
wb = xlrd.open_workbook('gdp_growth.xls', ignore_workbook_corruption=True)
ws = wb.sheet_by_index(0)

years = [int(ws.cell_value(0, c)) for c in range(1, ws.ncols)]

gdp_rows = []
gdp_country_names = {
    'Australia'     : 'Australia',
    'Canada'        : 'Canada',
    'Switzerland'   : 'Switzerland',
    'United Kingdom': 'United Kingdom',
    'Japan'         : 'Japan',
    'United States' : 'United States',
    'Euro area'     : 'Euro Area',
}

for i in range(ws.nrows):
    name = ws.cell_value(i, 0)
    if name in gdp_country_names:
        for j, year in enumerate(years):
            val = ws.cell_value(i, j + 1)
            if val != 'no data' and val != '':
                gdp_rows.append({
                    'Country'     : gdp_country_names[name],
                    'Year'        : year,
                    'GDP_Growth'  : float(val),
                    'Is_Forecast' : year >= 2025
                })

gdp = pd.DataFrame(gdp_rows)
gdp = gdp[(gdp['Year'] >= 2007) & (gdp['Year'] <= 2030)]
gdp['Date'] = pd.to_datetime(gdp['Year'].astype(str) + '-01-01')

# Expand to monthly
gdp_monthly = []
for country in COUNTRIES:
    df_c = (
        gdp[gdp['Country'] == country]
        .set_index('Date')[['GDP_Growth', 'Is_Forecast']]
        .reindex(monthly_range)
    )
    df_c['Country'] = country
    df_c['GDP_Growth']   = df_c['GDP_Growth'].ffill()
    df_c['Is_Forecast']  = df_c['Is_Forecast'].ffill().astype(bool)
    df_c.index.name = 'Date'
    gdp_monthly.append(df_c.reset_index())

gdp_monthly = pd.concat(gdp_monthly, ignore_index=True)

#  MERGE ALL DATASETS

In [ ]:
# Align dates to month-start for clean joining
for df in [inflation, rates]:
    df['Date'] = df['Date'].values.astype('datetime64[M]').astype('datetime64[ns]')

unemp_monthly['Date'] = unemp_monthly['Date'].values.astype('datetime64[M]').astype('datetime64[ns]')
gdp_monthly['Date']   = gdp_monthly['Date'].values.astype('datetime64[M]').astype('datetime64[ns]')

merged = (
    rates
    .merge(inflation,    on=['Country', 'Date'], how='outer')
    .merge(unemp_monthly, on=['Country', 'Date'], how='outer')
    .merge(gdp_monthly,   on=['Country', 'Date'], how='outer')
)

# Filter to our date range and countries
merged = merged[
    (merged['Country'].isin(COUNTRIES)) &
    (merged['Date'] >= '2007-01-01') &
    (merged['Date'] <= '2030-12-01')
].sort_values(['Country', 'Date']).reset_index(drop=True)

# EXPORT

In [ ]:
merged.to_csv('dashboard_data.csv', index=False)

print(f"Shape: {merged.shape}")
print(f"\nDate range: {merged['Date'].min()} to {merged['Date'].max()}")
print(f"\nCountries: {sorted(merged['Country'].unique())}")
print(f"\nColumns: {merged.columns.tolist()}")
print(f"\nMissing values:\n{merged.isnull().sum()}")
print(f"\nSample (United Kingdom, 2022):")
print(merged[(merged['Country'] == 'United Kingdom') & 
             (merged['Date'].dt.year == 2022)].head(6).to_string())

# colour palette 

In [ ]:
# Load data 
df = pd.read_csv('dashboard_data.csv', parse_dates=['Date'])

# Country order (Japan last, it's the outlier, draws the eye)
COUNTRIES = ['United States', 'United Kingdom', 'Euro Area',
             'Canada', 'Australia', 'Switzerland', 'Japan']

# Colourblind-safe qualitative palette (Brewer 2003)
# Based on ColorBrewer Set1 + Safe qualitative, 7 distinct hues
COUNTRY_COLOURS = {
    'United States'  : '#E41A1C',  # red
    'United Kingdom' : '#377EB8',  # blue
    'Euro Area'      : '#4DAF4A',  # green
    'Canada'         : '#FF7F00',  # orange
    'Australia'      : '#984EA3',  # purple
    'Switzerland'    : '#A65628',  # brown
    'Japan'          : '#999999',  # grey — deliberate: outlier = muted
}

# Year boundaries
YEAR_MIN = 2007
YEAR_MAX = 2030
FORECAST_START = 2025

# Quick check
print(f"Data loaded: {df.shape[0]:,} rows")
print(f"Countries:   {sorted(df['Country'].unique())}")
print(f"Date range:  {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Columns:     {df.columns.tolist()}")

# Data cleaning and pre-processing

In [ ]:

# Forward fill Japan's missing early policy rate months
df['Policy_Rate'] = df.groupby('Country')['Policy_Rate'].transform(
    lambda x: x.ffill().bfill()
)

# Add year column 
df['Year'] = df['Date'].dt.year

# Add month column 
df['YearMonth'] = df['Date'].dt.to_period('M').astype(str)

# Filter to forecast/non-forecast for chart styling
df_real     = df[df['Is_Forecast'] == False].copy()
df_forecast = df[df['Is_Forecast'] == True].copy()

# Annual aggregation 
df_annual = (
    df.groupby(['Country', 'Year'])
    .agg(
        Inflation    = ('Inflation',    'mean'),
        Policy_Rate  = ('Policy_Rate',  'mean'),
        Unemployment = ('Unemployment', 'mean'),
        GDP_Growth   = ('GDP_Growth',   'first'),
        Is_Forecast  = ('Is_Forecast',  'first')
    )
    .reset_index()
)

print(f"Missing Policy_Rate after fill: {df['Policy_Rate'].isnull().sum()}")
print(f"Missing Inflation remaining:    {df['Inflation'].isnull().sum()}")
print(f"Annual df shape: {df_annual.shape}")

# App layout

In [ ]:
app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.FLATLY],
    title='Monetary Policy & Inflation Dashboard'
)

# Colour helpers 
BG_MAIN    = '#F8F9FA'
BG_CARD    = '#FFFFFF'
TEXT_DARK  = '#2C3E50'
TEXT_MUTED = '#6C757D'
ACCENT     = '#2C7BB6'

# Reusable card wrapper 
def make_card(children, style=None):
    base = {
        'backgroundColor': BG_CARD,
        'borderRadius'   : '10px',
        'padding'        : '16px',
        'boxShadow'      : '0 1px 4px rgba(0,0,0,0.08)',
        'marginBottom'   : '16px'
    }
    if style:
        base.update(style)
    return html.Div(children, style=base)

# KPI card 
def kpi_card(title, value, subtitle, colour):
    return html.Div([
        html.P(title, style={
            'fontSize': '11px', 'fontWeight': '600',
            'color': TEXT_MUTED, 'marginBottom': '4px',
            'textTransform': 'uppercase', 'letterSpacing': '0.5px'
        }),
        html.H4(value, style={
            'fontSize': '22px', 'fontWeight': '700',
            'color': colour, 'marginBottom': '2px'
        }),
        html.P(subtitle, style={
            'fontSize': '11px', 'color': TEXT_MUTED, 'marginBottom': '0'
        })
    ], style={
        'backgroundColor': BG_CARD,
        'borderRadius'   : '10px',
        'padding'        : '16px 20px',
        'borderLeft'     : f'4px solid {colour}',
        'boxShadow'      : '0 1px 4px rgba(0,0,0,0.08)',
    })

# Layout 
app.layout = html.Div(style={'backgroundColor': BG_MAIN, 'minHeight': '100vh',
                              'fontFamily': 'Inter, Segoe UI, sans-serif'}, children=[

    # Header 
    html.Div(style={
        'background'   : 'linear-gradient(135deg, #1a3a5c 0%, #2C7BB6 100%)',
        'padding'      : '28px 40px 20px',
        'marginBottom' : '24px'
    }, children=[
        html.H1('Monetary Policy & The Inflation Crisis',
                style={'color': '#FFFFFF', 'fontWeight': '700',
                       'fontSize': '26px', 'marginBottom': '6px'}),
        html.P('How major central banks used interest rates to fight the 2021–2023 '
               'inflation crisis — and what happened to growth and unemployment.',
               style={'color': 'rgba(255,255,255,0.8)', 'fontSize': '14px',
                      'marginBottom': '20px'}),

        # Filters row 
        html.Div(style={'display': 'flex', 'gap': '32px',
                        'alignItems': 'flex-start', 'flexWrap': 'wrap'}, children=[

            # Country checklist
            html.Div([
                html.Label('Countries', style={
                    'color': 'rgba(255,255,255,0.7)', 'fontSize': '11px',
                    'fontWeight': '600', 'textTransform': 'uppercase',
                    'letterSpacing': '0.5px', 'marginBottom': '8px',
                    'display': 'block'
                }),
                dcc.Checklist(
                    id='country-filter',
                    options=[{'label': f'  {c}', 'value': c} for c in COUNTRIES],
                    value=COUNTRIES,
                    inline=True,
                    inputStyle={'marginRight': '4px', 'accentColor': '#F0C040'},
                    labelStyle={
                        'color': '#FFFFFF', 'fontSize': '13px',
                        'marginRight': '16px', 'cursor': 'pointer'
                    }
                )
            ]),

            # Year range slider
            html.Div(style={'minWidth': '280px'}, children=[
                html.Label('Year range', style={
                    'color': 'rgba(255,255,255,0.7)', 'fontSize': '11px',
                    'fontWeight': '600', 'textTransform': 'uppercase',
                    'letterSpacing': '0.5px', 'marginBottom': '8px',
                    'display': 'block'
                }),
                dcc.RangeSlider(
                    id='year-filter',
                    min=YEAR_MIN, max=YEAR_MAX,
                    value=[YEAR_MIN, YEAR_MAX],
                    marks={y: {'label': str(y),
                               'style': {'color': 'rgba(255,255,255,0.7)',
                                         'fontSize': '11px'}}
                           for y in range(2007, 2031, 3)},
                    step=1,
                    tooltip={'placement': 'bottom', 'always_visible': False}
                )
            ])
        ])
    ]),

    # Main content
    html.Div(style={'padding': '0 32px 32px'}, children=[

        # KPI row 
        html.Div(style={
            'display': 'grid',
            'gridTemplateColumns': 'repeat(4, 1fr)',
            'gap': '16px', 'marginBottom': '16px'
        }, children=[
            kpi_card('Peak Inflation',    '11.1%',  'United Kingdom · Oct 2022', '#E41A1C'),
            kpi_card('Peak Policy Rate',  '5.50%',  'United States · 2023',      '#377EB8'),
            kpi_card('Japan Outlier',     '0.10%',  'Rate held near-zero · 2023','#999999'),
            kpi_card('Fastest Hike Cycle','425 bps','Canada · Mar–Jan 2023',     '#FF7F00'),
        ]),
        
        # ── Narrative text box ────────────────────────────────────────────────
        html.Div(style={
            'backgroundColor' : '#EBF4FB',
            'borderLeft'      : '4px solid #2C7BB6',
            'borderRadius'    : '8px',
            'padding'         : '14px 20px',
            'marginBottom'    : '16px',
        }, children=[
            html.P(
                'Between 2021 and 2023, inflation across major economies surged '
                'to levels unseen since the 1980s, driven by post-COVID supply '
                'chain disruptions and unprecedented fiscal stimulus. Central banks '
                'responded with the fastest interest rate hiking cycle in four '
                'decades — yet outcomes diverged sharply. Japan, facing deflationary '
                'pressures rather than inflation, held rates near zero throughout, '
                'exposing how the same global shock produced radically different '
                'policy responses. Use the filters above to explore each economy\'s '
                'trajectory.',
                style={
                    'fontSize'    : '13px',
                    'color'       : '#1a3a5c',
                    'lineHeight'  : '1.7',
                    'marginBottom': '0'
                }
            )
        ]),

        # Section 2: Core story 
        make_card([
            html.H6('Inflation Rate vs Policy Rate (2007–2026)',
                    style={'color': TEXT_DARK, 'fontWeight': '600',
                           'marginBottom': '4px'}),
            html.P('Solid lines = inflation · Dashed lines = policy rate · '
                   'Shaded region = IMF forecast (2025–2030)',
                   style={'color': TEXT_MUTED, 'fontSize': '12px',
                          'marginBottom': '12px'}),
            dcc.Graph(id='line-chart-main', style={'height': '420px'},
                      config={'displayModeBar': False})
        ]),

        # Section 3: Country deep-dive 
        html.Div(style={
            'display': 'grid',
            'gridTemplateColumns': '1fr 1fr',
            'gap': '16px', 'marginBottom': '0'
        }, children=[
            make_card([
                html.H6('Inflation Intensity Heatmap',
                        style={'color': TEXT_DARK, 'fontWeight': '600',
                               'marginBottom': '4px'}),
                html.P('Average annual inflation by country and year',
                       style={'color': TEXT_MUTED, 'fontSize': '12px',
                              'marginBottom': '12px'}),
                dcc.Graph(id='heatmap-chart', style={'height': '300px'},
                          config={'displayModeBar': False})
            ]),
            make_card([
                html.H6('Policy Rate Hike Speed',
                        style={'color': TEXT_DARK, 'fontWeight': '600',
                               'marginBottom': '4px'}),
                html.P('Policy rate at pre-hike baseline, peak, and current level per country',
                        style={'color': TEXT_MUTED, 'fontSize': '12px', 'marginBottom': '12px'}),
                dcc.Graph(id='slope-chart', style={'height': '300px'},
                          config={'displayModeBar': False})
            ]),
        ]),

        # Section 4: Economic impact 
        html.Div(style={
            'display': 'grid',
            'gridTemplateColumns': '1fr 1fr',
            'gap': '16px', 'marginBottom': '0'
        }, children=[
            make_card([
                html.H6('GDP Growth Rate (2007–2030)',
                        style={'color': TEXT_DARK, 'fontWeight': '600',
                               'marginBottom': '4px'}),
                html.P('Real data (solid) · IMF forecast 2025–2030 (dashed)',
                       style={'color': TEXT_MUTED, 'fontSize': '12px',
                              'marginBottom': '12px'}),
                dcc.Graph(id='gdp-chart', style={'height': '300px'},
                          config={'displayModeBar': False})
            ]),
            make_card([
                html.H6('Unemployment Rate (2007–2026)',
                        style={'color': TEXT_DARK, 'fontWeight': '600',
                               'marginBottom': '4px'}),
                html.P('Unemployment rate: pre-COVID (2019) vs post-hike (2022) vs latest (2024)',
                        style={'color': TEXT_MUTED, 'fontSize': '12px', 'marginBottom': '12px'}),
                dcc.Graph(id='unemployment-chart', style={'height': '300px'},
                          config={'displayModeBar': False})
            ]),
        ]),

        # Section 5: Scatter 
        make_card([
            html.H6('Inflation vs Policy Rate — Annual Scatter',
                    style={'color': TEXT_DARK, 'fontWeight': '600',
                           'marginBottom': '4px'}),
            html.P('Each point = one country in one year · '
                   'Size = unemployment rate · Colour = country',
                   style={'color': TEXT_MUTED, 'fontSize': '12px',
                          'marginBottom': '12px'}),
            dcc.Graph(id='scatter-chart', style={'height': '380px'},
                      config={'displayModeBar': False})
        ]),

        # Footer 
        html.P([
            'Designed for policy analysts, financial journalists, and economics graduates. · ',
            'Sources: BIS (Policy Rates & CPI, 2026) · Our World in Data (Unemployment) · ',
            'IMF World Economic Outlook (GDP Growth, 2025) · ',
            'Forecasts shown as dashed lines · Colour-blind safe palette (Brewer, 2003)'
        ],
        style={
            'color'      : TEXT_MUTED,
            'fontSize'   : '11px',
            'textAlign'  : 'center',
            'marginTop'  : '8px',
            'borderTop'  : '1px solid #DEE2E6',
            'paddingTop' : '16px'
        }
        )
    ])
])

print("Layout defined")

# Chart builder functions

In [ ]:
# Shared layout defaults (applied to every chart) 
def base_layout(title=None):
    layout = dict(
        paper_bgcolor = 'rgba(0,0,0,0)',
        plot_bgcolor  = 'rgba(0,0,0,0)',
        font          = dict(family='Inter, Segoe UI, sans-serif',
                             color=TEXT_DARK, size=12),
        margin        = dict(l=48, r=24, t=36, b=48),
        legend        = dict(
            orientation = 'h',
            yanchor     = 'bottom',
            y           = 1.02,
            xanchor     = 'left',
            x           = 0,
            font        = dict(size=11),
            bgcolor     = 'rgba(0,0,0,0)'
        ),
        hoverlabel = dict(
            bgcolor    = '#2C3E50',
            font_color = '#FFFFFF',
            font_size  = 12,
            bordercolor= '#2C3E50'
        ),
        xaxis = dict(
            showgrid    = True,
            gridcolor   = '#EEEEEE',
            gridwidth   = 0.5,
            zeroline    = False,
            showline    = True,
            linecolor   = '#DDDDDD',
            tickfont    = dict(size=11)
        ),
        yaxis = dict(
            showgrid    = True,
            gridcolor   = '#EEEEEE',
            gridwidth   = 0.5,
            zeroline    = True,
            zerolinecolor = '#CCCCCC',
            zerolinewidth = 0.8,
            showline    = False,
            tickfont    = dict(size=11)
        ),
    )
    if title:
        layout['title'] = dict(text=title, font=dict(size=13), x=0, xanchor='left')
    return layout


# Key event annotations (reused across charts)
def key_annotations():
    # events = [
    #     ('2008-09-01', 'GFC',             '#888888', 1.01),
    #     ('2020-03-01', 'COVID',           '#888888', 1.08),
    #     ('2022-01-01', 'Inflation peak',  '#E41A1C', 1.01),
    #     ('2024-06-01', 'Rate cuts begin', '#377EB8', 1.08),
    # ]
    # shapes = []
    # annotations = []
    # for date, label, colour, y_pos in events:
    #     shapes.append(dict(
    #         type    = 'line',
    #         x0=date, x1=date, y0=0, y1=1,
    #         yref    = 'paper',
    #         line    = dict(color=colour, width=1, dash='dot'),
    #         opacity = 0.4
    #     ))
    #     annotations.append(dict(
    #         x        = date,
    #         y        = y_pos,
    #         yref     = 'paper',
    #         text     = label,
    #         showarrow= False,
    #         font     = dict(size=9, color=colour),
    #         xanchor  = 'left',
    #         opacity  = 0.85,
    #         bgcolor  = 'rgba(255,255,255,0.7)',
    #         borderpad= 2
    #     ))
    # return shapes, annotations
    return [], []
    


# Forecast shading helper
def forecast_shape():
    return dict(
        type      = 'rect',
        x0        = f'{FORECAST_START}-01-01',
        x1        = '2030-12-31',
        y0        = 0, y1 = 1,
        yref      = 'paper',
        fillcolor = '#F0F0F0',
        opacity   = 0.4,
        line      = dict(width=0)
    )

# ── Helper: hex colour → rgba string ─────────────────────────────────────────
def hex_to_rgba(hex_colour, alpha=0.1):
    hex_colour = hex_colour.lstrip('#')
    r, g, b = int(hex_colour[0:2], 16), int(hex_colour[2:4], 16), int(hex_colour[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

# ─────────────────────────────────────────────────────────────────────────────
# CHART 1 — Dual-axis line chart: Inflation vs Policy Rate
# ─────────────────────────────────────────────────────────────────────────────
def build_line_chart(selected_countries, year_range):
    dff = df[
        (df['Country'].isin(selected_countries)) &
        (df['Year'] >= year_range[0]) &
        (df['Year'] <= year_range[1])
    ].copy()

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    for country in selected_countries:
        c = dff[dff['Country'] == country]
        colour = COUNTRY_COLOURS[country]

        # Real data — inflation (solid, primary axis)
        c_real = c[c['Is_Forecast'] == False]
        fig.add_trace(go.Scatter(
            x    = c_real['Date'],
            y    = c_real['Inflation'],
            name = f'{country} · Inflation',
            mode = 'lines',
            line = dict(color=colour, width=1.8),
            hovertemplate = (
                f'<b>{country}</b><br>'
                'Inflation: %{y:.1f}%<br>'
                '%{x|%b %Y}<extra></extra>'
            )
        ), secondary_y=False)

        # Real data — policy rate (dashed, secondary axis)
        fig.add_trace(go.Scatter(
            x    = c_real['Date'],
            y    = c_real['Policy_Rate'],
            name = f'{country} · Rate',
            mode = 'lines',
            line = dict(color=colour, width=1.8, dash='dash'),
            hovertemplate = (
                f'<b>{country}</b><br>'
                'Policy rate: %{y:.2f}%<br>'
                '%{x|%b %Y}<extra></extra>'
            )
        ), secondary_y=True)

        # Forecast — policy rate (dotted)
        c_fore = c[c['Is_Forecast'] == True]
        if not c_fore.empty:
            fig.add_trace(go.Scatter(
                x    = c_fore['Date'],
                y    = c_fore['GDP_Growth'],
                name = f'{country} · Forecast',
                mode = 'lines',
                line = dict(color=colour, width=1.2, dash='dot'),
                opacity  = 0.5,
                showlegend = False,
                hovertemplate = (
                    f'<b>{country}</b><br>'
                    'GDP forecast: %{y:.1f}%<br>'
                    '%{x|%b %Y}<extra></extra>'
                )
            ), secondary_y=True)

    # Annotations + forecast shading
    shapes, annots = key_annotations()
    shapes.append(forecast_shape())

    # Forecast label
    annots.append(dict(
        x='2025-06-01', y=0.95, yref='paper',
        text='← IMF Forecast →',
        showarrow=False,
        font=dict(size=10, color='#999999'),
        xanchor='center'
    ))
    
    # Add 2% inflation target reference line
    shapes.append(dict(
        type      = 'line',
        x0        = '2007-01-01',
        x1        = '2030-12-31',
        y0        = 2, y1 = 2,
        yref      = 'y',
        line      = dict(color='#2CA02C', width=1.2, dash='dot'),
        opacity   = 0.7
    ))
    annots.append(dict(
        x         = '2007-06-01',
        y         = 2,
        yref      = 'y',
        text      = '2% target',
        showarrow = False,
        font      = dict(size=10, color='#2CA02C'),
        xanchor   = 'left',
        yanchor   = 'bottom',
        bgcolor   = 'rgba(255,255,255,0.7)',
        borderpad = 2,
        opacity   = 0.9
    ))

    layout = base_layout()
    layout.update(dict(
        shapes      = shapes,
        annotations = annots,
        yaxis       = dict(
            title     = 'Inflation rate (%)',
            showgrid  = True,
            gridcolor = '#EEEEEE',
            zeroline  = True,
            zerolinecolor = '#CCCCCC',
            ticksuffix = '%',
            tickfont  = dict(size=11)
        ),
        yaxis2      = dict(
            title     = 'Policy rate (%)',
            showgrid  = False,
            zeroline  = False,
            ticksuffix = '%',
            tickfont  = dict(size=11)
        ),
        hovermode   = 'x unified',
        legend      = dict(
            orientation = 'h',
            y=-0.18, x=0,
            font=dict(size=10),
            bgcolor='rgba(0,0,0,0)'
        )
    ))
    fig.update_layout(**layout)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# CHART 2 — Inflation heatmap (country × year)
# ─────────────────────────────────────────────────────────────────────────────
def build_heatmap(selected_countries, year_range):
    dff = df_annual[
        (df_annual['Country'].isin(selected_countries)) &
        (df_annual['Year'] >= year_range[0]) &
        (df_annual['Year'] <= year_range[1]) &
        (df_annual['Is_Forecast'] == False)
    ].copy()

    pivot = dff.pivot(index='Country', columns='Year', values='Inflation')
    pivot = pivot.reindex([c for c in COUNTRIES if c in selected_countries])

    fig = go.Figure(go.Heatmap(
        z             = pivot.values,
        x             = pivot.columns.astype(str),
        y             = pivot.index.tolist(),
        colorscale    = [
            [0.0,  '#2166AC'],   # deep blue  — deflation / very low
            [0.2,  '#92C5DE'],   # light blue — low inflation
            [0.4,  '#F7F7F7'],   # white      — ~2% target
            [0.65, '#FDDBC7'],   # light red  — elevated
            [0.85, '#D6604D'],   # red        — high
            [1.0,  '#B2182B'],   # dark red   — crisis level
        ],
        zmid          = 2,       # centre the scale at the 2% target
        zmin          = -2,
        zmax          = 12,
        colorbar      = dict(
            title      = dict(text='Inflation %', side='right',
                              font=dict(size=11)),
            thickness  = 12,
            len        = 0.8,
            ticksuffix = '%',
            tickfont   = dict(size=10)
        ),
        hovertemplate = (
            '<b>%{y}</b><br>'
            'Year: %{x}<br>'
            'Inflation: %{z:.1f}%<extra></extra>'
        ),
        text          = [[f'{v:.1f}' if not np.isnan(v) else ''
                          for v in row] for row in pivot.values],
        texttemplate  = '%{text}',
        textfont      = dict(size=9, color='#333333')
    ))

    layout = base_layout()
    layout.update(dict(
        xaxis  = dict(tickfont=dict(size=10), showgrid=False),
        yaxis  = dict(tickfont=dict(size=11), showgrid=False,
                      autorange='reversed'),
        margin = dict(l=120, r=60, t=20, b=40)
    ))
    fig.update_layout(**layout)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# CHART 3 — Grouped bar: Rate at baseline vs peak vs current (replaces slope)
# ─────────────────────────────────────────────────────────────────────────────
def build_slope_chart(selected_countries, year_range):
    # Pull annual averages for three key years
    key_years  = [2021, 2023, 2024]
    dff = df_annual[
        (df_annual['Country'].isin(selected_countries)) &
        (df_annual['Year'].isin(key_years)) &
        (df_annual['Is_Forecast'] == False)
    ].copy()

    labels = {2021: 'Pre-hike (2021)', 2023: 'Peak (2023)', 2024: 'Current (2024)'}
    bar_colours = ['#A8C8E8', '#E41A1C', '#4DAF4A']  # soft blue, red, green

    fig = go.Figure()

    for i, year in enumerate(key_years):
        dy = dff[dff['Year'] == year]
        # Maintain country order
        dy = dy.set_index('Country').reindex(
            [c for c in selected_countries if c in dy['Country'].values]
        ).reset_index()

        fig.add_trace(go.Bar(
            name        = labels[year],
            x           = dy['Country'],
            y           = dy['Policy_Rate'],
            marker_color= bar_colours[i],
            text        = dy['Policy_Rate'].apply(lambda v: f'{v:.2f}%'
                                                  if pd.notna(v) else ''),
            textposition= 'outside',
            textfont    = dict(size=10),
            hovertemplate = (
                '<b>%{x}</b><br>'
                f'{labels[year]}<br>'
                'Policy rate: %{y:.2f}%<extra></extra>'
            )
        ))

    layout = base_layout()
    layout.update(dict(
        barmode  = 'group',
        bargap   = 0.2,
        bargroupgap = 0.05,
        yaxis    = dict(
            title      = 'Policy rate (%)',
            ticksuffix = '%',
            showgrid   = True,
            gridcolor  = '#EEEEEE',
            zeroline   = True,
            zerolinecolor = '#CCCCCC',
            tickfont   = dict(size=11)
        ),
        xaxis    = dict(
            tickfont = dict(size=11),
            showgrid = False
        ),
        legend   = dict(
            orientation = 'h',
            y=1.12, x=0,
            font=dict(size=11),
            bgcolor='rgba(0,0,0,0)'
        ),
        margin   = dict(l=48, r=24, t=60, b=80)
    ))
    fig.update_layout(**layout)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# CHART 4 — GDP growth line chart (real + forecast)
# ─────────────────────────────────────────────────────────────────────────────
def build_gdp_chart(selected_countries, year_range):
    dff = df_annual[
        (df_annual['Country'].isin(selected_countries)) &
        (df_annual['Year'] >= year_range[0]) &
        (df_annual['Year'] <= year_range[1])
    ].copy()

    fig = go.Figure()

    for country in selected_countries:
        c      = dff[dff['Country'] == country]
        colour = COUNTRY_COLOURS[country]
        c_real = c[c['Is_Forecast'] == False]
        c_fore = c[c['Is_Forecast'] == True]

        fig.add_trace(go.Scatter(
            x    = c_real['Year'],
            y    = c_real['GDP_Growth'],
            name = country,
            mode = 'lines',
            line = dict(color=colour, width=2),
            hovertemplate = (
                f'<b>{country}</b><br>'
                'GDP growth: %{y:.1f}%<br>'
                'Year: %{x}<extra></extra>'
            )
        ))

        if not c_fore.empty:
            join = pd.concat([c_real.iloc[[-1]], c_fore])
            fig.add_trace(go.Scatter(
                x    = join['Year'],
                y    = join['GDP_Growth'],
                mode = 'lines',
                line = dict(color=colour, width=1.5, dash='dash'),
                opacity    = 0.6,
                showlegend = False,
                hovertemplate = (
                    f'<b>{country}</b><br>'
                    'GDP forecast: %{y:.1f}%<br>'
                    'Year: %{x}<extra></extra>'
                )
            ))

    # Zero reference line
    fig.add_hline(y=0, line_color='#AAAAAA', line_width=0.8, line_dash='dot')

    shapes, annots = key_annotations()
    shapes.append(forecast_shape())

    layout = base_layout()
    layout.update(dict(
        shapes      = shapes,
        annotations = annots,
        yaxis       = dict(
            title      = 'GDP growth (%)',
            ticksuffix = '%',
            showgrid   = True,
            gridcolor  = '#EEEEEE',
            zeroline   = True,
            zerolinecolor = '#CCCCCC',
            tickfont   = dict(size=11)
        ),
        hovermode = 'x unified'
    ))
    fig.update_layout(**layout)
    return fig



# ─────────────────────────────────────────────────────────────────────────────
# CHART 5 — Lollipop dot plot: Unemployment 2019 vs 2022 vs 2024 (replaces area)
# ─────────────────────────────────────────────────────────────────────────────
def build_unemployment_chart(selected_countries, year_range):
    key_years   = [2019, 2022, 2024]
    dot_colours = {
        2019: '#A8C8E8',  # soft blue  — pre-COVID baseline
        2022: '#E41A1C',  # red        — post-hike stress
        2024: '#4DAF4A',  # green      — recovery
    }
    labels = {2019: 'Pre-COVID (2019)', 2022: 'Post-hike (2022)', 2024: 'Latest (2024)'}

    dff = df_annual[
        (df_annual['Country'].isin(selected_countries)) &
        (df_annual['Year'].isin(key_years)) &
        (df_annual['Is_Forecast'] == False)
    ].copy()

    # Order countries consistently
    ordered = [c for c in COUNTRIES if c in selected_countries]

    fig = go.Figure()

    # Draw connector lines between 2019 and 2024 per country (lollipop stem)
    for country in ordered:
        c = dff[dff['Country'] == country]
        vals = {}
        for _, row in c.iterrows():
            vals[row['Year']] = row['Unemployment']

        if 2019 in vals and 2024 in vals:
            fig.add_trace(go.Scatter(
                x         = [vals[2019], vals[2024]],
                y         = [country, country],
                mode      = 'lines',
                line      = dict(color='#CCCCCC', width=2),
                showlegend= False,
                hoverinfo = 'skip'
            ))

    # Draw dots for each year
    for year in key_years:
        dy = dff[dff['Year'] == year].copy()
        dy = dy.set_index('Country').reindex(ordered).reset_index()

        fig.add_trace(go.Scatter(
            x    = dy['Unemployment'],
            y    = dy['Country'],
            mode = 'markers',
            name = labels[year],
            marker = dict(
                color  = dot_colours[year],
                size   = 14,
                line   = dict(color='white', width=1.5)
            ),
            hovertemplate = (
                '<b>%{y}</b><br>'
                f'{labels[year]}<br>'
                'Unemployment: %{x:.1f}%<extra></extra>'
            )
        ))

    layout = base_layout()
    layout.update(dict(
        xaxis = dict(
            title      = 'Unemployment rate (%)',
            ticksuffix = '%',
            showgrid   = True,
            gridcolor  = '#EEEEEE',
            zeroline   = False,
            tickfont   = dict(size=11)
        ),
        yaxis = dict(
            showgrid    = False,
            tickfont    = dict(size=11),
            autorange   = 'reversed',
            categoryorder = 'array',
            categoryarray = list(reversed(ordered))
        ),
        legend = dict(
            orientation = 'h',
            y=1.12, x=0,
            font=dict(size=11),
            bgcolor='rgba(0,0,0,0)'
        ),
        margin = dict(l=130, r=24, t=60, b=48),
        hovermode = 'closest'
    ))
    fig.update_layout(**layout)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# CHART 6 — Animated scatter: Inflation vs Policy Rate
# ─────────────────────────────────────────────────────────────────────────────
def build_scatter_chart(selected_countries, year_range):
    dff = df_annual[
        (df_annual['Country'].isin(selected_countries)) &
        (df_annual['Year'] >= year_range[0]) &
        (df_annual['Year'] <= year_range[1]) &
        (df_annual['Is_Forecast'] == False) &
        (df_annual['Inflation'].notna()) &
        (df_annual['Policy_Rate'].notna())
    ].copy()

    fig = px.scatter(
        dff,
        x           = 'Policy_Rate',
        y           = 'Inflation',
        color       = 'Country',
        size        = 'Unemployment',
        animation_frame = 'Year',
        hover_name  = 'Country',
        hover_data  = {
            'Policy_Rate'  : ':.2f',
            'Inflation'    : ':.1f',
            'Unemployment' : ':.1f',
            'Year'         : True,
            'Country'      : False
        },
        color_discrete_map = COUNTRY_COLOURS,
        size_max    = 28,
        labels      = {
            'Policy_Rate'  : 'Policy rate (%)',
            'Inflation'    : 'Inflation rate (%)',
            'Unemployment' : 'Unemployment (%)'
        }
    )

    # 45-degree reference line (rate = inflation → real rate = 0)
    max_val = max(dff['Policy_Rate'].max(), dff['Inflation'].max()) + 1
    fig.add_trace(go.Scatter(
        x    = [0, max_val],
        y    = [0, max_val],
        mode = 'lines',
        line = dict(color='#CCCCCC', width=1, dash='dot'),
        showlegend    = False,
        hoverinfo     = 'skip'
    ))

    # Annotation on reference line
    fig.add_annotation(
        x=max_val * 0.8, y=max_val * 0.8,
        text='Rate = Inflation<br>(real rate = 0)',
        showarrow=False,
        font=dict(size=9, color='#AAAAAA'),
        xanchor='left'
    )

    layout = base_layout()
    layout.update(dict(
        xaxis = dict(
            title      = 'Policy rate (%)',
            ticksuffix = '%',
            showgrid   = True,
            gridcolor  = '#EEEEEE',
            zeroline   = True,
            zerolinecolor = '#CCCCCC',
            tickfont   = dict(size=11)
        ),
        yaxis = dict(
            title      = 'Inflation rate (%)',
            ticksuffix = '%',
            showgrid   = True,
            gridcolor  = '#EEEEEE',
            zeroline   = True,
            zerolinecolor = '#CCCCCC',
            tickfont   = dict(size=11)
        ),
        legend = dict(
            orientation = 'v',
            x=1.02, y=1,
            font=dict(size=11)
        )
    ))
    fig.update_layout(**layout)

    # Style animation buttons
    fig.update_layout(
        updatemenus=[dict(
            type       = 'buttons',
            showactive = False,
            y          = -0.9,
            x          = 0.5,
            xanchor    = 'center',
            yanchor    = 'top',
            pad        = dict(t=10),
            buttons    = [
                dict(label='▶  Play',
                     method='animate',
                     args=[None, dict(
                         frame=dict(duration=600, redraw=True),
                         fromcurrent=True
                     )]),
                dict(label='⏸  Pause',
                     method='animate',
                     args=[[None], dict(
                         frame=dict(duration=0, redraw=False),
                         mode='immediate'
                     )])
            ]
        )],
            sliders=[{
                'currentvalue': {
                    'prefix'   : 'Year: ',
                    'visible'  : True,
                    'xanchor'  : 'center',
                    'font'     : {'size': 12, 'color': TEXT_MUTED}
                },
                'pad'        : {'t': 50, 'b': 10},
                'x'          : 0.1,
                'len'        : 0.8,
                'xanchor'    : 'left',
            }]
        )
    
    return fig


print("All chart functions defined")

# Callbacks

In [ ]:
@callback(
    Output('line-chart-main',    'figure'),
    Output('heatmap-chart',      'figure'),
    Output('slope-chart',        'figure'),
    Output('gdp-chart',          'figure'),
    Output('unemployment-chart', 'figure'),
    Output('scatter-chart',      'figure'),
    Input('country-filter',      'value'),
    Input('year-filter',         'value'),
)
def update_all_charts(selected_countries, year_range):
    # Fallback — if user deselects everything, show all
    if not selected_countries:
        selected_countries = COUNTRIES

    return (
        build_line_chart(selected_countries, year_range),
        build_heatmap(selected_countries, year_range),
        build_slope_chart(selected_countries, year_range),
        build_gdp_chart(selected_countries, year_range),
        build_unemployment_chart(selected_countries, year_range),
        build_scatter_chart(selected_countries, year_range),
    )

print("Callbacks registered")

# Run the app

In [ ]:
# http://127.0.0.1:8050
if __name__ == '__main__':
    app.run(debug=True, port=8050)

# Export dashboard as standalone HTML

In [ ]:
import plotly.io as pio
import json

# Reset plotly template to default before building figures for export
# Avoids conflict with dbc.themes.FLATLY set during app initialisation
pio.templates.default = 'plotly_white'

all_countries = COUNTRIES
all_years     = [2007, 2030]

# Build all figures
figures = [
    ('main-line',    'Inflation Rate vs Policy Rate',
     'Solid = inflation · Dashed = policy rate · Shaded region = IMF forecast (2025–2030)',
     build_line_chart(all_countries, all_years),         'full'),
    ('heatmap',      'Inflation Intensity Heatmap',
     'Average annual inflation by country and year',
     build_heatmap(all_countries, all_years),            'half'),
    ('slope',        'Policy Rate: Baseline vs Peak vs Current',
     'Grouped bar — pre-hike (2021) · peak (2023) · current (2024)',
     build_slope_chart(all_countries, all_years),        'half'),
    ('gdp',          'GDP Growth Rate (2007–2030)',
     'Real data (solid) · IMF forecast 2025–2030 (dashed)',
     build_gdp_chart(all_countries, all_years),          'half'),
    ('unemployment', 'Unemployment: Before & After Rate Hikes',
     'Dot plot — pre-COVID (2019) · post-hike (2022) · latest (2024)',
     build_unemployment_chart(all_countries, all_years), 'half'),
    ('scatter',      'Inflation vs Policy Rate — Animated Scatter',
     'Each point = one country × one year · Size = unemployment rate · Press Play to animate',
     build_scatter_chart(all_countries, all_years),      'full'),
]

# Serialise figures to JSON
fig_jsons = {fid: pio.to_json(fig) for fid, _, _, fig, _ in figures}

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Monetary Policy & Inflation Dashboard</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{
    font-family: Inter, 'Segoe UI', sans-serif;
    background: #F8F9FA;
    color: #2C3E50;
  }}

  /* ── Header ── */
  .header {{
    background: linear-gradient(135deg, #1a3a5c 0%, #2C7BB6 100%);
    padding: 28px 40px 24px;
    color: white;
  }}
  .header h1 {{ font-size: 26px; font-weight: 700; margin-bottom: 6px; }}
  .header p  {{ font-size: 14px; opacity: 0.85; margin-bottom: 20px; }}

  /* ── Filter bar ── */
  .filter-bar {{
    background: rgba(255,255,255,0.12);
    border-radius: 10px;
    padding: 14px 20px;
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    gap: 16px;
  }}
  .filter-label {{
    font-size: 11px;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    color: rgba(255,255,255,0.7);
    white-space: nowrap;
  }}
  .filter-divider {{
    width: 1px;
    height: 28px;
    background: rgba(255,255,255,0.25);
  }}
  .country-toggles {{
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
  }}
  .toggle-btn {{
    padding: 5px 14px;
    border-radius: 20px;
    border: 1.5px solid rgba(255,255,255,0.5);
    background: transparent;
    color: white;
    font-size: 12px;
    cursor: pointer;
    transition: all 0.15s;
    font-family: inherit;
  }}
  .toggle-btn.active {{
    background: rgba(255,255,255,0.25);
    border-color: white;
    font-weight: 600;
  }}
  .toggle-btn:hover {{ background: rgba(255,255,255,0.15); }}

	/* ── Year dual-handle slider ── */
		.year-filter {{
			display: flex;
			align-items: center;
			gap: 12px;
			flex: 1;
			min-width: 260px;
		}}
		.year-label {{
			font-size: 12px;
			color: rgba(255,255,255,0.9);
			font-weight: 600;
			min-width: 36px;
			text-align: center;
			white-space: nowrap;
		}}
		.slider-wrapper {{
			position: relative;
			flex: 1;
			height: 20px;
			display: flex;
			align-items: center;
		}}
		.slider-track {{
			position: absolute;
			width: 100%;
			height: 4px;
			background: rgba(255,255,255,0.2);
			border-radius: 2px;
			pointer-events: none;
		}}
		.slider-range {{
			position: absolute;
			height: 4px;
			background: #F0C040;
			border-radius: 2px;
			pointer-events: none;
		}}
		.slider-wrapper input[type=range] {{
			position: absolute;
			width: 100%;
			height: 4px;
			background: transparent;
			appearance: none;
			-webkit-appearance: none;
			pointer-events: none;
			margin: 0;
		}}
		.slider-wrapper input[type=range]::-webkit-slider-thumb {{
			appearance: none;
			-webkit-appearance: none;
			width: 16px;
			height: 16px;
			border-radius: 50%;
			background: #F0C040;
			border: 2px solid white;
			cursor: pointer;
			pointer-events: all;
			box-shadow: 0 1px 3px rgba(0,0,0,0.3);
		}}
		.slider-wrapper input[type=range]::-moz-range-thumb {{
			width: 16px;
			height: 16px;
			border-radius: 50%;
			background: #F0C040;
			border: 2px solid white;
			cursor: pointer;
			pointer-events: all;
			box-shadow: 0 1px 3px rgba(0,0,0,0.3);
		}}

  /* ── KPI row ── */
  .kpis {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 16px;
    padding: 24px 32px 8px;
  }}
  .kpi {{
    background: white;
    border-radius: 10px;
    padding: 16px 20px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.08);
  }}
  .kpi-label {{
    font-size: 11px;
    font-weight: 600;
    color: #6C757D;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    margin-bottom: 4px;
  }}
  .kpi-value {{
    font-size: 22px;
    font-weight: 700;
    margin-bottom: 2px;
  }}
  .kpi-sub {{ font-size: 11px; color: #6C757D; }}

  /* ── Content grid ── */
  .content {{ padding: 0 32px 32px; }}
  .grid-2 {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 16px;
    margin-bottom: 16px;
  }}
  .full-width {{ margin-bottom: 16px; }}

  /* ── Chart card ── */
  .card {{
    background: white;
    border-radius: 10px;
    padding: 16px;
    box-shadow: 0 1px 4px rgba(0,0,0,0.08);
    position: relative;
  }}
  .card-title {{
    font-size: 14px;
    font-weight: 600;
    color: #2C3E50;
    margin-bottom: 2px;
  }}
  .card-sub {{
    font-size: 12px;
    color: #6C757D;
    margin-bottom: 10px;
  }}

  /* ── Expand button ── */
  .expand-btn {{
    position: absolute;
    top: 12px;
    right: 12px;
    background: #F0F4F8;
    border: none;
    border-radius: 6px;
    padding: 4px 8px;
    font-size: 11px;
    color: #6C757D;
    cursor: pointer;
    font-family: inherit;
    transition: background 0.15s;
    z-index: 10;
  }}
  .expand-btn:hover {{ background: #DEE2E6; color: #2C3E50; }}

  /* ── Fullscreen modal ── */
  .modal-overlay {{
    display: none;
    position: fixed;
    inset: 0;
    background: rgba(0,0,0,0.7);
    z-index: 1000;
    align-items: center;
    justify-content: center;
  }}
  .modal-overlay.open {{ display: flex; }}
  .modal-box {{
    background: white;
    border-radius: 12px;
    width: 95vw;
    height: 90vh;
    padding: 20px;
    position: relative;
    display: flex;
    flex-direction: column;
  }}
  .modal-header {{
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 12px;
  }}
  .modal-title {{ font-size: 16px; font-weight: 600; }}
  .modal-close {{
    background: #F0F4F8;
    border: none;
    border-radius: 6px;
    padding: 6px 12px;
    font-size: 13px;
    cursor: pointer;
    font-family: inherit;
  }}
  .modal-close:hover {{ background: #DEE2E6; }}
  #modal-chart {{ flex: 1; min-height: 0; }}

  /* ── Footer ── */
  .footer {{
    text-align: center;
    font-size: 11px;
    color: #6C757D;
    padding: 16px 32px 32px;
    border-top: 1px solid #DEE2E6;
    margin: 8px 32px 0;
  }}
</style>
</head>
<body>

<!-- ── Header + filters ── -->
<div class="header">
  <h1>Monetary Policy &amp; The Inflation Crisis</h1>
  <p>How major central banks used interest rates to fight the 2021–2023 inflation crisis
     — and what happened to growth and unemployment.</p>
  <div class="filter-bar">

    <!-- Country toggles -->
    <span class="filter-label">Countries</span>
    <div class="country-toggles" id="country-toggles"></div>

    <div class="filter-divider"></div>

    <!-- Year dual-handle slider -->
    <span class="filter-label">Years</span>
    <div class="year-filter">
      <span class="year-label" id="year-min-label">2007</span>
      <div class="slider-wrapper">
        <div class="slider-track"></div>
        <div class="slider-range" id="slider-range"></div>
        <input type="range" id="year-min" min="2007" max="2030"
               value="2007" step="1" oninput="onYearChange('min')">
        <input type="range" id="year-max" min="2007" max="2030"
               value="2030" step="1" oninput="onYearChange('max')">
      </div>
      <span class="year-label" id="year-max-label">2030</span>
    </div>

  </div>
</div>

<!-- ── KPI cards ── -->
<div class="kpis">
  <div class="kpi" style="border-left:4px solid #E41A1C">
    <div class="kpi-label">Peak Inflation</div>
    <div class="kpi-value" style="color:#E41A1C">11.1%</div>
    <div class="kpi-sub">United Kingdom · Oct 2022</div>
  </div>
  <div class="kpi" style="border-left:4px solid #377EB8">
    <div class="kpi-label">Peak Policy Rate</div>
    <div class="kpi-value" style="color:#377EB8">5.50%</div>
    <div class="kpi-sub">United States · 2023</div>
  </div>
  <div class="kpi" style="border-left:4px solid #999999">
    <div class="kpi-label">Japan Outlier</div>
    <div class="kpi-value" style="color:#999999">0.10%</div>
    <div class="kpi-sub">Rate held near-zero · 2023</div>
  </div>
  <div class="kpi" style="border-left:4px solid #FF7F00">
    <div class="kpi-label">Fastest Hike Cycle</div>
    <div class="kpi-value" style="color:#FF7F00">425 bps</div>
    <div class="kpi-sub">Canada · Mar 2022–Jan 2023</div>
  </div>
</div>

<!-- ── Narrative text box ── -->
<div style="margin: 8px 32px 0; padding: 14px 20px;
            background: #EBF4FB; border-left: 4px solid #2C7BB6;
            border-radius: 8px;">
  <p style="font-size:13px; color:#1a3a5c; line-height:1.7; margin:0">
    Between 2021 and 2023, inflation across major economies surged to levels
    unseen since the 1980s, driven by post-COVID supply chain disruptions and
    unprecedented fiscal stimulus. Central banks responded with the fastest
    interest rate hiking cycle in four decades — yet outcomes diverged sharply.
    Japan, facing deflationary pressures rather than inflation, held rates near
    zero throughout, exposing how the same global shock produced radically
    different policy responses. Use the filters above to explore each economy's
    trajectory.
  </p>
</div>

<!-- ── Charts ── -->
<div class="content">

  <!-- Full width: Main line chart -->
  <div class="full-width">
    <div class="card">
      <div class="card-title">Inflation Rate vs Policy Rate</div>
      <div class="card-sub">Solid = inflation · Dashed = policy rate · Shaded region = IMF forecast (2025–2030)</div>
      <button class="expand-btn" onclick="expandChart('main-line','Inflation Rate vs Policy Rate')">⤢ Expand</button>
      <div id="main-line" style="height:420px"></div>
    </div>
  </div>

  <!-- 2-col: Heatmap + Bar chart -->
  <div class="grid-2">
    <div class="card">
      <div class="card-title">Inflation Intensity Heatmap</div>
      <div class="card-sub">Average annual inflation by country and year</div>
      <button class="expand-btn" onclick="expandChart('heatmap','Inflation Intensity Heatmap')">⤢ Expand</button>
      <div id="heatmap" style="height:320px"></div>
    </div>
    <div class="card">
      <div class="card-title">Policy Rate: Baseline vs Peak vs Current</div>
      <div class="card-sub">Grouped bar — pre-hike (2021) · peak (2023) · current (2024)</div>
      <button class="expand-btn" onclick="expandChart('slope','Policy Rate Comparison')">⤢ Expand</button>
      <div id="slope" style="height:320px"></div>
    </div>
  </div>

  <!-- 2-col: GDP + Dot plot -->
  <div class="grid-2">
    <div class="card">
      <div class="card-title">GDP Growth Rate (2007–2030)</div>
      <div class="card-sub">Real data (solid) · IMF forecast 2025–2030 (dashed)</div>
      <button class="expand-btn" onclick="expandChart('gdp','GDP Growth Rate')">⤢ Expand</button>
      <div id="gdp" style="height:320px"></div>
    </div>
    <div class="card">
      <div class="card-title">Unemployment: Before &amp; After Rate Hikes</div>
      <div class="card-sub">Dot plot — pre-COVID (2019) · post-hike (2022) · latest (2024)</div>
      <button class="expand-btn" onclick="expandChart('unemployment','Unemployment Dot Plot')">⤢ Expand</button>
      <div id="unemployment" style="height:320px"></div>
    </div>
  </div>

  <!-- Full width: Scatter -->
  <div class="full-width">
    <div class="card">
      <div class="card-title">Inflation vs Policy Rate — Animated Scatter</div>
      <div class="card-sub">Each point = one country × one year · Size = unemployment rate · Press Play to animate</div>
      <button class="expand-btn" onclick="expandChart('scatter','Animated Scatter')">⤢ Expand</button>
      <div id="scatter" style="height:480px"></div>
    </div>
  </div>

</div>

<!-- ── Footer ── -->
<div class="footer">
  Designed for policy analysts, financial journalists, and economics graduates. ·
  Sources: BIS (Policy Rates &amp; CPI, 2026) · Our World in Data (Unemployment) ·
  IMF World Economic Outlook (GDP Growth, 2025) ·
  Forecasts shown as dashed lines · Colour-blind safe palette (Brewer, 2003)
</div>

<!-- ── Fullscreen modal ── -->
<div class="modal-overlay" id="modal-overlay" onclick="closeModal(event)">
  <div class="modal-box">
    <div class="modal-header">
      <span class="modal-title" id="modal-title"></span>
      <button class="modal-close" onclick="closeModalDirect()">✕ Close</button>
    </div>
    <div id="modal-chart"></div>
  </div>
</div>

<script>
// ── Constants ─────────────────────────────────────────────────────────────────
const COUNTRIES = {json.dumps(COUNTRIES)};
const COLOURS   = {json.dumps(COUNTRY_COLOURS)};
const FILTERABLE_COUNTRY = ['main-line', 'slope', 'gdp', 'unemployment', 'scatter'];
const FILTERABLE_YEAR    = ['main-line', 'gdp'];

// ── State ─────────────────────────────────────────────────────────────────────
let activeCountries = new Set(COUNTRIES);
let yearMin = 2007;
let yearMax = 2030;

// ── Store original figure data for year filtering ─────────────────────────────
const figData    = {json.dumps(fig_jsons)};
const origTraces = {{}};  // divId → deep copy of original traces

// ── Render all charts ─────────────────────────────────────────────────────────
const chartConfig = {{
  responsive             : true,
  displayModeBar         : true,
  doubleClick            : 'reset+autosize',
  scrollZoom             : true,
  modeBarButtonsToRemove : ['lasso2d', 'select2d'],
}};

Object.entries(figData).forEach(([divId, jsonStr]) => {{
  const fig = JSON.parse(jsonStr);
  // Pass frames separately for animated charts (scatter)
  Plotly.newPlot(divId, fig.data, fig.layout, chartConfig).then(() => {{
    if (fig.frames && fig.frames.length > 0) {{
      Plotly.addFrames(divId, fig.frames);
    }}
  }});
  // Store deep copy of original traces
  origTraces[divId] = JSON.parse(JSON.stringify(fig.data));
}});

// ── Country toggles ───────────────────────────────────────────────────────────
function renderToggles() {{
  const container = document.getElementById('country-toggles');
  container.innerHTML = '';
  COUNTRIES.forEach(c => {{
    const btn = document.createElement('button');
    btn.className = 'toggle-btn' + (activeCountries.has(c) ? ' active' : '');
    btn.textContent = c;
    btn.style.borderColor = activeCountries.has(c)
      ? COLOURS[c] : 'rgba(255,255,255,0.4)';
    btn.style.background = activeCountries.has(c)
      ? COLOURS[c] + '55' : 'transparent';
    btn.onclick = () => toggleCountry(c);
    container.appendChild(btn);
  }});
}}

function toggleCountry(country) {{
  if (activeCountries.has(country)) {{
    if (activeCountries.size === 1) return;
    activeCountries.delete(country);
  }} else {{
    activeCountries.add(country);
  }}
  renderToggles();
  applyCountryFilter();
}}

function applyCountryFilter() {{
  FILTERABLE_COUNTRY.forEach(divId => {{
    const el = document.getElementById(divId);
    if (!el || !el.data) return;
    const visible = el.data.map(trace => {{
      if (!trace.name) return true;
      const match = COUNTRIES.find(c => trace.name.startsWith(c));
      return match ? activeCountries.has(match) : true;
    }});
    Plotly.restyle(divId, {{ visible }});
  }});
}}

// ── Year range filter ─────────────────────────────────────────────────────────
function onYearChange(handle) {{
  let minVal = parseInt(document.getElementById('year-min').value);
  let maxVal = parseInt(document.getElementById('year-max').value);

  if (minVal >= maxVal) {{
    if (handle === 'min') {{
      minVal = maxVal - 1;
      document.getElementById('year-min').value = minVal;
    }} else {{
      maxVal = minVal + 1;
      document.getElementById('year-max').value = maxVal;
    }}
  }}

  yearMin = minVal;
  yearMax = maxVal;

  document.getElementById('year-min-label').textContent = yearMin;
  document.getElementById('year-max-label').textContent = yearMax;

  // Update coloured track between handles
  const total = 2030 - 2007;
  const leftPct  = ((yearMin - 2007) / total) * 100;
  const rightPct = ((2030 - yearMax) / total) * 100;
  document.getElementById('slider-range').style.left  = leftPct  + '%';
  document.getElementById('slider-range').style.right = rightPct + '%';

  applyYearFilter();
}}

function applyYearFilter() {{
  FILTERABLE_YEAR.forEach(divId => {{
    const el = document.getElementById(divId);
    if (!el || !el.data) return;
    const orig = origTraces[divId];
    if (!orig) return;

    const xUpdates = [];
    const yUpdates = [];

    orig.forEach(trace => {{
      if (!trace.x || !trace.x.length) {{
        xUpdates.push(trace.x);
        yUpdates.push(trace.y);
        return;
      }}

      const isDate = typeof trace.x[0] === 'string' && trace.x[0].includes('-');
      const isYear = typeof trace.x[0] === 'number';

      if (isDate) {{
        const xi = [], yi = [];
        trace.x.forEach((d, i) => {{
          const yr = new Date(d).getFullYear();
          if (yr >= yearMin && yr <= yearMax) {{
            xi.push(d);
            yi.push(trace.y[i]);
          }}
        }});
        xUpdates.push(xi);
        yUpdates.push(yi);
      }} else if (isYear) {{
        const xi = [], yi = [];
        trace.x.forEach((yr, i) => {{
          if (yr >= yearMin && yr <= yearMax) {{
            xi.push(yr);
            yi.push(trace.y[i]);
          }}
        }});
        xUpdates.push(xi);
        yUpdates.push(yi);
      }} else {{
        xUpdates.push(trace.x);
        yUpdates.push(trace.y);
      }}
    }});

    Plotly.restyle(divId, {{ x: xUpdates, y: yUpdates }});
  }});
}}

// ── Expand to fullscreen modal ────────────────────────────────────────────────
let modalChartId = null;

function expandChart(divId, title) {{
  modalChartId = divId;
  document.getElementById('modal-title').textContent = title;
  document.getElementById('modal-overlay').classList.add('open');

  const source = document.getElementById(divId);
  const modal  = document.getElementById('modal-chart');
  modal.style.height = (window.innerHeight * 0.75) + 'px';

  const fig = {{
    data  : JSON.parse(JSON.stringify(source.data)),
    layout: JSON.parse(JSON.stringify(source.layout)),
    frames: source._transitionData && source._transitionData._frames
            ? JSON.parse(JSON.stringify(source._transitionData._frames))
            : [],
  }};
  fig.layout.height   = undefined;
  fig.layout.autosize = true;

  Plotly.newPlot('modal-chart', fig.data, fig.layout, {{
    ...chartConfig,
    displayModeBar: true,
  }}).then(() => {{
    if (fig.frames && fig.frames.length > 0) {{
      Plotly.addFrames('modal-chart', fig.frames);
    }}
  }});

  // Sync country filter into modal
  const visible = fig.data.map(trace => {{
    if (!trace.name) return true;
    const match = COUNTRIES.find(c => trace.name.startsWith(c));
    return match ? activeCountries.has(match) : true;
  }});
  Plotly.restyle('modal-chart', {{ visible }});
}}

function closeModal(event) {{
  if (event.target === document.getElementById('modal-overlay')) {{
    closeModalDirect();
  }}
}}

function closeModalDirect() {{
  document.getElementById('modal-overlay').classList.remove('open');
  Plotly.purge('modal-chart');
  modalChartId = null;
}}

// ── Initialise ────────────────────────────────────────────────────────────────
(function() {{
  const total = 2030 - 2007;
  const leftPct  = ((yearMin - 2007) / total) * 100;
  const rightPct = ((2030 - yearMax)  / total) * 100;
  document.getElementById('slider-range').style.left  = leftPct  + '%';
  document.getElementById('slider-range').style.right = rightPct + '%';
}})();

renderToggles();
</script>
</body>
</html>"""

with open('dashboard.html', 'w', encoding='utf-8') as f:
    f.write(html)

print("dashboard.html exported")